# Examen final
### Análisis de dataset de imágenes MRI cerebrales

Este proyecto analizará 40.000 imágenes IRM cerebrales para poder entrenar un modelo de regresión logistica* el cual predicirá, dados los datos extraídos de una imagen, si un paciente tiene alzheimers o no.

Para esto, se necesitará:
- Obtener el dataset desde Kaggle
- Transformar datos de imágenes a datos en csv, obteniendo toda la información relevante 
- Agrupar los csv en un DataFrame
- Entrenar y probar un modelo de regresión lineal para predecir si un paciente tiene alzheimers o no, y que tan severo es. 

### Importación

In [1]:
%pip install pandas
%pip install matplotlib
%pip install kagglehub
%pip install numpy
%pip install stats

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.


https://www.kaggle.com/datasets/lukechugh/best-alzheimer-mri-dataset-99-accuracy

In [48]:
import kagglehub
# descargar a la misma carpeta:
kagglehub.dataset_download("uraninjo/augmented-alzheimer-mri-dataset", output_dir='Datos')

100%|██████████| 380M/380M [00:20<00:00, 19.3MB/s] 

Extracting files...


'Datos'

El unico "problema" de este dataset es que los MRIs de pacientes con Alzheimers pueden ser bastante dificiles de encontrar.

A los pacientes con esta enfermedad no les agrada estar en una maquina ruidosa por varias horas, y el diagnostico usualmente es relativamente obvio. 

In [3]:
import pandas as pd
import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Tenemos los datos, pero no podemos usarlos todavia dado que nuestras herramientas no incluyen librerías como tensorflow que puedan trabajar con imágenes.

Primero, debemos definir que parte de cada imagen es que. Debemos definir que parte es:
- Fluido cerebroespinal
- Materia gris
- Materia blanca

En cada imagen. 

Para esto, usaremos Pillow. Sin embargo, debemos también entender que no existen rangos específicos universales para la escala de grises que usaremos para extraer las características necesarias de las imágenes.

Entonces primero que todo, debemos nosotros mismos investigar que rango correspondería a este dataset.

Para analizar las imágenes, también necesitamos OpenCV (cv2) con la cual podemos leer imagenes con escala gris.

In [20]:
!pip install opencv-python==4.8.1.78

Para sacar los limites, analizaremos los minimos locales de cada imagen en las carpetas para encontrar limites de FCE y MG, y de MG y MB. Luego, se van a promediar estos para encontrar una buena aproximacion para los limites necesarios.

In [ ]:
import os
from PIL import Image
import cv2

listaClases = ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']

listaPixeles = []
limites = []

for i in listaClases:
    DATASET_DIR = f"./Datos/OriginalDataset/{i}"
    sample_files = os.listdir(DATASET_DIR)

    all_pixels = []

    for file in sample_files:
        if file.endswith('.jpg'):
            img = cv2.imread(os.path.join(DATASET_DIR, file), cv2.IMREAD_GRAYSCALE) # leer imagen en grayscale con cv2
            pixels = img[img > 10].flatten() 
            all_pixels.extend(pixels)
            counts, _ = np.histogram(pixels, bins=256, range=(0, 255))
            window_size = 11
            smoothed = np.convolve(counts, np.ones(window_size)/window_size, mode='same')
            csf_gm_valley = 40 + np.argmin(smoothed[40:80])
            gm_wm_valley = 160 + np.argmin(smoothed[160:200])
            limites.append((csf_gm_valley, gm_wm_valley))
    # Plot the distribution
    plt.figure(figsize=(10, 5))
    plt.hist(all_pixels, bins=256, range=(0, 255), color='gray', alpha=0.75)
    plt.title(f"Distribucion de pixeles en categoria {i}")
    plt.xlabel("Pixel Value (0-255)")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()

: 

In [23]:
print(limites)
meanValleys = np.mean(limites, axis=0)

print(meanValleys)

[(44, 184), (55, 189), (55, 188), (51, 177), (64, 168), (56, 191), (46, 183), (69, 175), (51, 175), (45, 198), (78, 178), (51, 181), (51, 162), (78, 164), (62, 162), (43, 188), (54, 191), (56, 181), (41, 189), (53, 178), (51, 168), (50, 182), (55, 185), (67, 174), (57, 168), (42, 175), (42, 184), (48, 181), (79, 191), (75, 176), (52, 181), (55, 182), (46, 177), (49, 173), (62, 181), (57, 178), (41, 181), (78, 179), (48, 191), (52, 187), (49, 198), (41, 171), (48, 188), (40, 177), (50, 185), (44, 184), (78, 188), (55, 161), (60, 165), (68, 160), (50, 181), (40, 189), (62, 179), (52, 186), (45, 176), (79, 179), (66, 186), (46, 176), (58, 169), (51, 181), (56, 184), (55, 173), (78, 172), (61, 184), (75, 160), (42, 176), (64, 174), (40, 186), (66, 187), (51, 175), (53, 178), (75, 178), (51, 176), (47, 180), (50, 186), (78, 184), (51, 181), (41, 183), (45, 190), (64, 173), (42, 184), (43, 178), (51, 173), (67, 173), (65, 170), (78, 185), (49, 177), (55, 189), (48, 193), (47, 188), (51, 179)

De esto, podemos tener que los limites estan entre 0, 60, 180, 255.

Por lo tanto, cuando hablamos de valores de escala gris:

- Fluido cerebroespinal: 0-59
- Materia gris: 60-180
- Materia blanca: 181-255

## Investigación

Ahora, con los datos que tenemos, queremos ver como podemos usarlos para predecir Alzheimers en una persona. 

Que sabemos del Alzheimers?

- El cerebro se encoge en general.
- Provoca disminución de materia gris y materia blanca.
- Al disminuir la materia gris y la materia blanca, el fluido cerebroespinal rellena los vacíos dejados por la ausencia de ambas materias.

No podemos simplemente decir que si una imagen tiene menos pixeles, entonces es mas posible que la persona tenga Alzheimers, ya que el cerebro de cada uno es muy distinto al del otro. Cada uno tiene un cráneo o cerebro de tamaño diferente, y eso no es indicación de Alzheimers o de mayor/menor capacidad cognitiva.

Lo que si podemos usar es el BV/CSF Index.

$$BV/CSF = \frac{MG\ Total + MB\ Total}{FCE\ Total}$$

Con:

- MG = Materia Gris
- MB = Materia Blanca
- FCE = Fluido Cerebroespinal




Este índice es usado en neuroimagenología para medir el volumen del cerebro, y es una variable que podemos usar para nuestra regresión logistica.

Para esto, debemos calcular el índice para cada una de nuestras imágenes.

In [49]:
# Iterar sobre cada clase
imagePixels = [] # Inicializa lista de datos que se convertirá en diccionario

for i in listaClases:
    DATASET_DIR = f"./Datos/AugmentedAlzheimerDataset/{i}"
    sample_files = os.listdir(DATASET_DIR)[:100]



    for file in sample_files:
        if file.endswith('.jpg'):
            img = cv2.imread(os.path.join(DATASET_DIR, file), cv2.IMREAD_GRAYSCALE) # leer imagen en grayscale con cv2
            pixels = img[img > 10].flatten() 
            all_pixels.extend(pixels)
            counts, _ = np.histogram(pixels, bins=256, range=(0, 255))
            window_size = 11
            smoothed = np.convolve(counts, np.ones(window_size)/window_size, mode='same')
            FCE = int(smoothed[0:60].sum())
            MG = int(smoothed[60:181].sum())
            MB = int(smoothed[181:255].sum())
            BVCSFIndex = ((MG + MB)/FCE)
            # Nuestros rangos son: 
            # Fluido Cerebroespinal: 0-59
            # Materia gris: 60-180
            # Materia blanca: 181-255
            imagePixels.append({
                "IDImagen": file,
                "Fluido Cerebroespinal": FCE, # 0 a 59
                "Materia Gris": MG, # 60 a 180
                "Materia Blanca": MB, # 181 a 255
                "BV/CSF Index": BVCSFIndex,
                "Clase": i
            })
            

Ahora, tenemos un diccionario con datos correspondientes a cada imagen. 

In [50]:
df = pd.DataFrame(imagePixels)
df

,IDImagen,Fluido Cerebroespinal,Materia Gris,Materia Blanca,BV/CSF Index,Clase
0,00005576-2b76-44ca-8572-a5057201433f.jpg,1993,11721,4648,8.213246,NonDemented
1,0007d7c8-609f-4339-b810-727535907c42.jpg,20436,9793,7653,0.853690,NonDemented
2,00126637-9d0c-4261-b79d-b6897ba3f4af.jpg,3594,8290,6849,4.212298,NonDemented
3,0016c2ee-28b3-4e43-988a-09192f039c8d.jpg,17993,7867,9333,0.955927,NonDemented
4,0018f6e9-4716-4f8d-aad6-b9bc1b40b255.jpg,2890,8465,8211,5.770242,NonDemented
...,...,...,...,...,...,...
295,031ea6f4-47b8-485c-a469-1822ff31a185.jpg,20743,14636,2620,0.831895,MildDemented
296,03415644-104e-49fc-b157-27f4c18658cd.jpg,3379,11421,6694,5.361054,MildDemented
297,034ca06f-bab6-4266-bd26-dc6f4f34fd72.jpg,21612,7273,7771,0.696095,MildDemented
298,03587048-4287-4e30-b47e-f97c1040ce0f.jpg,19335,14002,4662,0.965296,MildDemented


Aca mismo es donde decidi cambiar el proyecto a uno de regresion logística, pero igual podremos ver los resultados de la regresión lineal si fuera necesario. 